# 03 - Documentación del Pipeline y Plan de Producción

---

## Del Notebook a Producción

Este es el **notebook final** del proyecto de Ciencia de Datos aplicado a la gestión inteligente de flotas vehiculares. Su propósito es documentar completamente el pipeline construido a lo largo de 6 fases y establecer un plan claro para la transición a producción.

### La Brecha Notebook-Producción

La mayoría de los proyectos de Data Science mueren en la etapa de prototipo. La documentación adecuada es el puente entre un análisis exploratorio exitoso y un sistema que genera valor real para el negocio:

- **Reproducibilidad**: que cualquier persona pueda replicar los resultados
- **Mantenibilidad**: que el sistema pueda evolucionar sin depender de una sola persona
- **Escalabilidad**: que la arquitectura soporte el crecimiento de datos y usuarios
- **Monitoreo**: que se detecten degradaciones del modelo y anomalías en los datos

---

## Diagrama del Pipeline

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    PIPELINE DE CIENCIA DE DATOS                            │
│                    Inteligencia de Flota Vehicular                         │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                           │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐            │
│  │  DATOS   │    │ LIMPIEZA │    │ FEATURES │    │ MODELOS  │            │
│  │  CRUDOS  │───>│    Y     │───>│    Y     │───>│   ML     │            │
│  │          │    │ TRANSF.  │    │ SELECCIÓN│    │          │            │
│  └──────────┘    └──────────┘    └──────────┘    └──────────┘            │
│       │               │               │               │                  │
│  telemetry_*.csv  valores nulos   features_ml.csv  RandomForest          │
│  fleet_profiles   outliers        vehicle_survey   KMeans                │
│  buyer_surveys    encoding        _merged.csv      IsolationForest       │
│                   scaling                                                │
│                                                                          │
│  ┌──────────┐    ┌──────────────┐    ┌──────────────────┐               │
│  │PREDICCIÓN│    │  DASHBOARD   │    │  DOCUMENTACIÓN   │               │
│  │    Y     │───>│     Y        │───>│       Y          │               │
│  │ANOMALÍAS │    │ REPORTES     │    │  PLAN PRODUCCIÓN │               │
│  └──────────┘    └──────────────┘    └──────────────────┘               │
│       │               │                    │                             │
│  predicciones    Matplotlib/Plotly     Este notebook                     │
│  alertas         resumen ejecutivo     arquitectura                      │
│  z-scores        KPIs interactivos     reproducibilidad                 │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [1]:
# === Configuración ===
import os
import glob
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

print(f"Raíz del proyecto: {project_root}")
print(f"Fecha de documentación: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Python: {sys.version}")

Raíz del proyecto: /home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos
Fecha de documentación: 2026-03-15 01:11
Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


---

## Inventario de Datos

Catálogo completo de todos los archivos de datos del proyecto, con rutas, tamaños y descripciones.

In [2]:
# === Inventario de Datos ===

def get_file_info(filepath, project_root):
    """Obtener información de un archivo."""
    rel_path = os.path.relpath(filepath, project_root)
    size_bytes = os.path.getsize(filepath)
    if size_bytes > 1024 * 1024:
        size_str = f"{size_bytes / (1024*1024):.1f} MB"
    elif size_bytes > 1024:
        size_str = f"{size_bytes / 1024:.1f} KB"
    else:
        size_str = f"{size_bytes} B"
    return rel_path, size_str, size_bytes

# Buscar todos los archivos de datos
data_patterns = [
    os.path.join(project_root, 'data', '**', '*.csv'),
    os.path.join(project_root, 'data', '**', '*.parquet'),
    os.path.join(project_root, 'data', '**', '*.json'),
]

data_files = []
for pattern in data_patterns:
    data_files.extend(glob.glob(pattern, recursive=True))

# Descripciones conocidas
descriptions = {
    'fleet_profiles.csv': 'Perfiles de vehículos de la flota (tipo, características)',
    'buyer_surveys.csv': 'Encuestas de satisfacción de compradores (500 registros)',
    'features_ml.csv': 'Features procesadas para modelos de ML',
    'vehicle_survey_merged.csv': 'Datos de vehículos y encuestas fusionados',
}

inventory = []
for f in sorted(data_files):
    rel_path, size_str, size_bytes = get_file_info(f, project_root)
    basename = os.path.basename(f)
    desc = descriptions.get(basename, '')
    if 'telemetry_' in basename:
        desc = f'Telemetría vehicular (timestamp, sensores, GPS)'
    inventory.append({
        'Ruta': rel_path,
        'Tamaño': size_str,
        'Bytes': size_bytes,
        'Descripción': desc
    })

df_inventory = pd.DataFrame(inventory)

print(f"Total de archivos de datos: {len(df_inventory)}")
print(f"Tamaño total: {df_inventory['Bytes'].sum() / (1024*1024):.1f} MB")
print("\n" + "=" * 100)
print(df_inventory[['Ruta', 'Tamaño', 'Descripción']].to_string(index=False))
print("=" * 100)

Total de archivos de datos: 34
Tamaño total: 966.5 MB

                                       Ruta  Tamaño                                               Descripción
             data/processed/features_ml.csv 10.8 KB                    Features procesadas para modelos de ML
   data/processed/vehicle_survey_merged.csv 15.6 KB                 Datos de vehículos y encuestas fusionados
                data/raw/fleet_profiles.csv  5.1 KB Perfiles de vehículos de la flota (tipo, características)
         data/raw/surveys/buyer_surveys.csv 84.0 KB  Encuestas de satisfacción de compradores (500 registros)
data/raw/telemetry/telemetry_2025-01-01.csv 27.2 MB           Telemetría vehicular (timestamp, sensores, GPS)
data/raw/telemetry/telemetry_2025-01-02.csv 34.0 MB           Telemetría vehicular (timestamp, sensores, GPS)
data/raw/telemetry/telemetry_2025-01-03.csv 32.0 MB           Telemetría vehicular (timestamp, sensores, GPS)
data/raw/telemetry/telemetry_2025-01-04.csv 30.9 MB           Tel

---

## Inventario de Modelos

Catálogo de los modelos entrenados, incluyendo algoritmo, target y métricas de rendimiento.

In [3]:
# === Inventario de Modelos ===

models_dir = os.path.join(project_root, 'models')
model_files = []

if os.path.exists(models_dir):
    for pattern in ['*.pkl', '*.joblib', '*.json', '*.h5']:
        model_files.extend(glob.glob(os.path.join(models_dir, '**', pattern), recursive=True))

print(f"Archivos de modelos encontrados: {len(model_files)}")

model_inventory = []
for mf in sorted(model_files):
    rel_path, size_str, size_bytes = get_file_info(mf, project_root)
    basename = os.path.basename(mf)
    
    model_info = {
        'Archivo': rel_path,
        'Tamaño': size_str,
        'Algoritmo': 'N/A',
        'Target': 'N/A',
        'Métricas': 'N/A'
    }
    
    # Intentar cargar con joblib para obtener info del modelo
    if mf.endswith(('.pkl', '.joblib')):
        try:
            import joblib
            model = joblib.load(mf)
            algo_name = type(model).__name__
            model_info['Algoritmo'] = algo_name
            
            # Intentar obtener parámetros
            if hasattr(model, 'n_estimators'):
                model_info['Algoritmo'] += f' (n={model.n_estimators})'
            if hasattr(model, 'n_clusters'):
                model_info['Algoritmo'] += f' (k={model.n_clusters})'
        except Exception as e:
            model_info['Algoritmo'] = f'Error al cargar: {str(e)[:50]}'
    
    elif mf.endswith('.json'):
        try:
            with open(mf, 'r') as f:
                meta = json.load(f)
            model_info['Algoritmo'] = meta.get('algorithm', 'N/A')
            model_info['Target'] = meta.get('target', 'N/A')
            if 'metrics' in meta:
                metrics_str = ', '.join([f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" 
                                         for k, v in meta['metrics'].items()])
                model_info['Métricas'] = metrics_str
        except Exception:
            pass
    
    model_inventory.append(model_info)

if model_inventory:
    df_models = pd.DataFrame(model_inventory)
    print("\n" + df_models.to_string(index=False))
else:
    print("\nNo se encontraron archivos de modelos en el directorio models/.")
    print("Los modelos se entrenan dinámicamente en los notebooks de las fases 4-6.")

Archivos de modelos encontrados: 1

                          Archivo   Tamaño Algoritmo Target Métricas
models/regression_consumption.pkl 109.9 KB      dict    N/A      N/A


---

## Inventario de Notebooks

Catálogo completo de todos los notebooks del proyecto, organizados por fase.

In [4]:
# === Inventario de Notebooks ===

notebooks_dir = os.path.join(project_root, 'notebooks')
notebook_files = sorted(glob.glob(os.path.join(notebooks_dir, '**', '*.ipynb'), recursive=True))

# Mapeo de fases a conceptos clave
phase_concepts = {
    'phase1': 'Exploración y análisis descriptivo',
    'phase2': 'Limpieza, transformación y feature engineering',
    'phase3': 'Análisis estadístico y pruebas de hipótesis',
    'phase4': 'Machine Learning supervisado y no supervisado',
    'phase5': 'Detección de anomalías y series temporales',
    'phase6': 'Síntesis ejecutiva y documentación'
}

nb_inventory = []
for nb in notebook_files:
    rel_path = os.path.relpath(nb, project_root)
    basename = os.path.basename(nb)
    
    # Detectar fase
    phase = 'N/A'
    for p in phase_concepts:
        if p in nb:
            phase = p.replace('phase', 'Fase ')
            break
    
    # Extraer título del notebook
    title = basename.replace('.ipynb', '').replace('_', ' ').title()
    try:
        with open(nb, 'r', encoding='utf-8') as f:
            nb_content = json.load(f)
        # Buscar el primer markdown con título
        for cell in nb_content.get('cells', []):
            if cell['cell_type'] == 'markdown':
                source = ''.join(cell['source'])
                for line in source.split('\n'):
                    if line.startswith('# '):
                        title = line.replace('# ', '').strip()
                        break
                if title != basename.replace('.ipynb', '').replace('_', ' ').title():
                    break
    except Exception:
        pass
    
    # Contar celdas
    try:
        n_cells = len(nb_content.get('cells', []))
        n_code = sum(1 for c in nb_content['cells'] if c['cell_type'] == 'code')
        n_md = sum(1 for c in nb_content['cells'] if c['cell_type'] == 'markdown')
    except Exception:
        n_cells = n_code = n_md = 0
    
    nb_inventory.append({
        'Fase': phase,
        'Archivo': basename,
        'Título': title[:60],
        'Celdas': f'{n_code}c + {n_md}md = {n_cells}',
    })

df_notebooks = pd.DataFrame(nb_inventory)
print(f"Total de notebooks: {len(df_notebooks)}")
print("\n" + "=" * 110)
print(df_notebooks.to_string(index=False))
print("=" * 110)

Total de notebooks: 31

  Fase                               Archivo                                                     Título          Celdas
Fase 1           01_numpy_fundamentals.ipynb 01 - Fundamentos de NumPy: Señales de Telemetría Vehicular  13c + 9md = 22
Fase 1          02_pandas_fundamentals.ipynb       02 - Fundamentos de Pandas: DataFrames de Telemetría  10c + 8md = 18
Fase 1            03_vehicle_simulator.ipynb                       03 - Simulador de Vehículos Completo   9c + 6md = 15
Fase 1             04_survey_simulator.ipynb                 04 - Simulador de Encuestas de Compradores   8c + 6md = 14
Fase 1                 05_data_merging.ipynb                       05 - Merge de Telemetría y Encuestas  10c + 7md = 17
Fase 2         01_univariate_telemetry.ipynb                     01 - Análisis Univariado de Telemetría   8c + 7md = 15
Fase 2            02_temporal_patterns.ipynb                                   02 - Patrones Temporales   7c + 7md = 14
Fase 2          

---

## Reproducibilidad

Instrucciones para reproducir el entorno y los resultados del proyecto.

In [5]:
# === Reproducibilidad ===

print("=" * 70)
print("INFORMACIÓN DEL ENTORNO")
print("=" * 70)
print(f"Python: {sys.version}")
print(f"Plataforma: {sys.platform}")
print(f"Directorio del proyecto: {project_root}")
print()

# Verificar requirements.txt
req_path = os.path.join(project_root, 'requirements.txt')
if os.path.exists(req_path):
    print("Contenido de requirements.txt:")
    print("-" * 40)
    with open(req_path, 'r') as f:
        print(f.read())
else:
    print("requirements.txt no encontrado.")
    print("\nDependencias principales del proyecto:")
    print("-" * 40)
    deps = [
        'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly',
        'scikit-learn', 'scipy', 'joblib', 'jupyter'
    ]
    for dep in deps:
        try:
            mod = __import__(dep.replace('-', '_').split('[')[0])
            version = getattr(mod, '__version__', 'instalado')
            print(f"  {dep}: {version}")
        except ImportError:
            print(f"  {dep}: NO INSTALADO")

print()
print("=" * 70)
print("INSTRUCCIONES DE CONFIGURACIÓN")
print("=" * 70)
print("""
1. Clonar el repositorio:
   git clone <url-del-repo>
   cd Ciencia_Datos

2. Crear entorno virtual:
   python -m venv .venv
   source .venv/bin/activate  # Linux/Mac
   .venv\\Scripts\\activate   # Windows

3. Instalar dependencias:
   pip install -r requirements.txt

4. Ejecutar notebooks en orden:
   jupyter notebook notebooks/phase1/01_*.ipynb
   # ... continuar con cada fase

5. Verificar datos:
   ls data/raw/telemetry/       # Archivos de telemetría
   ls data/raw/                 # fleet_profiles.csv
   ls data/raw/surveys/         # buyer_surveys.csv
""")

INFORMACIÓN DEL ENTORNO
Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
Plataforma: linux
Directorio del proyecto: /home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos

Contenido de requirements.txt:
----------------------------------------
numpy>=1.24
pandas>=2.0
jupyter>=1.0
jupyterlab>=4.0
matplotlib>=3.7
seaborn>=0.12
plotly>=5.15
folium>=0.14
scipy>=1.11
statsmodels>=0.14
scikit-learn>=1.3
xgboost>=2.0
prophet>=1.1
nbconvert>=7.0


INSTRUCCIONES DE CONFIGURACIÓN

1. Clonar el repositorio:
   git clone <url-del-repo>
   cd Ciencia_Datos

2. Crear entorno virtual:
   python -m venv .venv
   source .venv/bin/activate  # Linux/Mac
   .venv\Scripts\activate   # Windows

3. Instalar dependencias:
   pip install -r requirements.txt

4. Ejecutar notebooks en orden:
   jupyter notebook notebooks/phase1/01_*.ipynb
   # ... continuar con cada fase

5. Verificar datos:
   ls data/raw/telemetry/       # Archivos de telemetrí

---

## Arquitectura de Producción Propuesta

La siguiente arquitectura describe cómo llevar este proyecto de notebooks a un sistema en producción robusto y escalable.

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                     ARQUITECTURA DE PRODUCCIÓN PROPUESTA                       │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                               │
│  ┌────────────┐     ┌────────────┐     ┌────────────────┐                     │
│  │ Vehículos  │     │            │     │                │                     │
│  │   (IoT)    │────>│   Kafka    │────>│   Consumer     │                     │
│  │ Sensores   │     │  Broker    │     │   Service      │                     │
│  │ GPS/OBD-II │     │            │     │                │                     │
│  └────────────┘     └────────────┘     └───────┬────────┘                     │
│                                                │                              │
│                                                v                              │
│                                        ┌───────────────┐                      │
│                                        │  PostgreSQL   │                      │
│                                        │  + TimescaleDB│                      │
│                                        │  (telemetría) │                      │
│                                        └───────┬───────┘                      │
│                                                │                              │
│                     ┌──────────────────────────┼──────────────────────┐        │
│                     │                          │                      │        │
│                     v                          v                      v        │
│           ┌──────────────┐           ┌──────────────┐      ┌──────────────┐   │
│           │   FastAPI    │           │   Airflow    │      │   React      │   │
│           │   + ML API   │           │  Scheduler   │      │  Dashboard   │   │
│           │  (predicción │           │  (ETL diario │      │  (frontend)  │   │
│           │   anomalías) │           │   retraining)│      │              │   │
│           └──────────────┘           └──────────────┘      └──────────────┘   │
│                     │                          │                      │        │
│                     └──────────────────────────┼──────────────────────┘        │
│                                                │                              │
│                                                v                              │
│                               ┌────────────────────────────┐                  │
│                               │  Prometheus + Grafana      │                  │
│                               │  + Loki (monitoreo)        │                  │
│                               └────────────────────────────┘                  │
│                                                                               │
│                               ┌────────────────────────────┐                  │
│                               │  Docker + Kubernetes       │                  │
│                               │  (orquestación)            │                  │
│                               └────────────────────────────┘                  │
│                                                                               │
└─────────────────────────────────────────────────────────────────────────────────┘
```

### Descripción de Componentes

| Componente | Qué hace | Por qué | Alternativa |
|------------|----------|---------|-------------|
| **Kafka** | Ingesta de datos en streaming desde sensores IoT | Manejo de alto volumen, baja latencia, tolerancia a fallos | RabbitMQ, AWS Kinesis, Google Pub/Sub |
| **Consumer Service** | Procesa mensajes de Kafka, valida y transforma datos | Desacopla la ingesta del almacenamiento | Apache Flink, Spark Streaming |
| **PostgreSQL + TimescaleDB** | Almacena telemetría como series temporales | Consultas temporales optimizadas, SQL estándar | InfluxDB, Cassandra, ClickHouse |
| **FastAPI + ML** | API REST para predicciones y detección de anomalías | Async, rápido, documentación automática (OpenAPI) | Flask, Django REST, gRPC |
| **Airflow** | Orquesta ETL diario y reentrenamiento de modelos | Programación de tareas, dependencias, monitoreo | Prefect, Dagster, Luigi |
| **React Dashboard** | Frontend interactivo para stakeholders | Componentes reutilizables, ecosistema maduro | Vue.js, Streamlit, Dash |
| **Prometheus + Grafana** | Monitoreo de métricas del sistema y modelos | Estándar de la industria para observabilidad | Datadog, New Relic |
| **Loki** | Centralización de logs | Integración nativa con Grafana | ELK Stack, Splunk |
| **Docker + K8s** | Contenedorización y orquestación | Portabilidad, escalado automático, alta disponibilidad | Docker Compose (dev), ECS, Cloud Run |

In [6]:
# === Verificación del estado actual del proyecto ===

print("=" * 70)
print("VERIFICACIÓN DEL ESTADO DEL PROYECTO")
print("=" * 70)

# Verificar estructura de directorios
expected_dirs = [
    'data/raw/telemetry',
    'data/raw/surveys',
    'data/processed',
    'models',
    'notebooks/phase1',
    'notebooks/phase2',
    'notebooks/phase3',
    'notebooks/phase4',
    'notebooks/phase5',
    'notebooks/phase6',
]

print("\nEstructura de directorios:")
for d in expected_dirs:
    full_path = os.path.join(project_root, d)
    exists = os.path.exists(full_path)
    n_files = len(os.listdir(full_path)) if exists else 0
    status = 'OK' if exists else 'FALTA'
    print(f"  [{status}] {d}/ ({n_files} archivos)")

# Verificar archivos clave
expected_files = [
    'data/raw/fleet_profiles.csv',
    'data/raw/surveys/buyer_surveys.csv',
    'data/processed/features_ml.csv',
    'data/processed/vehicle_survey_merged.csv',
]

print("\nArchivos clave:")
for f in expected_files:
    full_path = os.path.join(project_root, f)
    if os.path.exists(full_path):
        size = os.path.getsize(full_path)
        print(f"  [OK] {f} ({size/1024:.1f} KB)")
    else:
        print(f"  [FALTA] {f}")

# Contar archivos de telemetría
tel_files = glob.glob(os.path.join(project_root, 'data/raw/telemetry/telemetry_*.csv'))
print(f"\nArchivos de telemetría: {len(tel_files)}")

VERIFICACIÓN DEL ESTADO DEL PROYECTO

Estructura de directorios:
  [OK] data/raw/telemetry/ (30 archivos)
  [OK] data/raw/surveys/ (1 archivos)
  [OK] data/processed/ (2 archivos)
  [OK] models/ (1 archivos)
  [OK] notebooks/phase1/ (6 archivos)
  [OK] notebooks/phase2/ (7 archivos)
  [OK] notebooks/phase3/ (6 archivos)
  [OK] notebooks/phase4/ (7 archivos)
  [OK] notebooks/phase5/ (7 archivos)
  [OK] notebooks/phase6/ (4 archivos)

Archivos clave:
  [OK] data/raw/fleet_profiles.csv (5.1 KB)
  [OK] data/raw/surveys/buyer_surveys.csv (84.0 KB)
  [OK] data/processed/features_ml.csv (10.8 KB)
  [OK] data/processed/vehicle_survey_merged.csv (15.6 KB)

Archivos de telemetría: 30


---

## Fases de Implementación

Plan de despliegue progresivo en 5 fases a lo largo de 5 meses.

### Fase 1: Fundamentos (Mes 1)
- Configurar infraestructura base: PostgreSQL + TimescaleDB
- Crear pipeline ETL básico con Airflow para ingesta de CSV
- Containerizar con Docker
- **Entregable**: Base de datos poblada con datos históricos

### Fase 2: API y Modelos (Mes 2)
- Desplegar FastAPI con endpoints de predicción
- Serializar modelos con MLflow
- Crear endpoints: `/predict/consumption`, `/detect/anomaly`, `/classify/risk`
- Tests unitarios y de integración
- **Entregable**: API documentada y funcionando

### Fase 3: Streaming (Mes 3)
- Implementar Kafka para ingesta en tiempo real
- Consumer service con validación de datos
- Alertas en tiempo real para anomalías
- **Entregable**: Pipeline de streaming funcional

### Fase 4: Dashboard y Monitoreo (Mes 4)
- Dashboard Grafana con paneles principales
- Alertas configuradas (Slack, email)
- Prometheus para métricas del sistema
- Loki para centralización de logs
- **Entregable**: Dashboard operativo con alertas

### Fase 5: Escalado y Optimización (Mes 5)
- Migrar a Kubernetes para escalado automático
- Implementar CI/CD para modelos (MLOps)
- A/B testing para nuevos modelos
- Documentación de operaciones (runbooks)
- **Entregable**: Sistema en producción completo

In [7]:
# === Timeline visual de implementación ===

timeline_data = {
    'Fase': ['Fundamentos', 'API y Modelos', 'Streaming', 'Dashboard', 'Escalado'],
    'Mes': [1, 2, 3, 4, 5],
    'Componentes': [
        'PostgreSQL, Docker, Airflow ETL',
        'FastAPI, MLflow, Tests',
        'Kafka, Consumer, Alertas RT',
        'Grafana, Prometheus, Loki',
        'Kubernetes, CI/CD, MLOps'
    ],
    'Riesgo': ['Bajo', 'Medio', 'Alto', 'Medio', 'Alto'],
    'Personas': [1, 2, 2, 1, 2]
}

df_timeline = pd.DataFrame(timeline_data)

print("=" * 80)
print("PLAN DE IMPLEMENTACIÓN - 5 MESES")
print("=" * 80)
print(df_timeline.to_string(index=False))
print("=" * 80)
print(f"\nPersonas-mes total: {df_timeline['Personas'].sum()}")
print(f"Costo estimado (a $5,000/persona-mes): ${df_timeline['Personas'].sum() * 5000:,}")

PLAN DE IMPLEMENTACIÓN - 5 MESES
         Fase  Mes                     Componentes Riesgo  Personas
  Fundamentos    1 PostgreSQL, Docker, Airflow ETL   Bajo         1
API y Modelos    2          FastAPI, MLflow, Tests  Medio         2
    Streaming    3     Kafka, Consumer, Alertas RT   Alto         2
    Dashboard    4       Grafana, Prometheus, Loki  Medio         1
     Escalado    5        Kubernetes, CI/CD, MLOps   Alto         2

Personas-mes total: 8
Costo estimado (a $5,000/persona-mes): $40,000


---

## Lecciones Aprendidas

Reflexiones clave de cada fase del proyecto.

### Fase 1: Exploración
**Lección**: Nunca subestimar el tiempo dedicado a entender los datos. La exploración visual inicial reveló patrones que guiaron todo el análisis posterior. Invertir en EDA ahorra semanas en fases posteriores.

### Fase 2: Limpieza y Transformación
**Lección**: El 80% del trabajo de un científico de datos es preparación de datos. Crear pipelines de limpieza reproducibles desde el inicio evita "deuda técnica" que se acumula exponencialmente.

### Fase 3: Análisis Estadístico
**Lección**: Las pruebas estadísticas formales dan credibilidad a los hallazgos. No basta con "ver" una diferencia en un gráfico; hay que cuantificar la significancia y el tamaño del efecto.

### Fase 4: Machine Learning
**Lección**: El modelo más complejo no siempre es el mejor. Un Random Forest bien tunado puede superar a modelos más sofisticados cuando los datos son limitados. La interpretabilidad es tan importante como la precisión.

### Fase 5: Anomalías y Series Temporales
**Lección**: La detección de anomalías requiere contexto de negocio. Un z-score > 2.5 no siempre es una anomalía real; la validación con expertos del dominio es esencial para reducir falsos positivos.

### Fase 6: Síntesis y Comunicación
**Lección**: Los hallazgos técnicos solo generan valor cuando se comunican efectivamente a los tomadores de decisiones. Un buen resumen ejecutivo con impacto cuantificado vale más que cien notebooks técnicos.

In [8]:
# === Resumen del proyecto ===

print("=" * 70)
print("RESUMEN FINAL DEL PROYECTO")
print("=" * 70)

# Contar todos los notebooks
all_notebooks = glob.glob(os.path.join(project_root, 'notebooks', '**', '*.ipynb'), recursive=True)
all_data = glob.glob(os.path.join(project_root, 'data', '**', '*.*'), recursive=True)
all_data = [f for f in all_data if os.path.isfile(f)]
all_models = glob.glob(os.path.join(project_root, 'models', '**', '*.*'), recursive=True)

# Calcular tamaño total
total_size = sum(os.path.getsize(f) for f in all_data if os.path.exists(f))

print(f"""
Proyecto: Inteligencia de Flota Vehicular
{'─' * 50}

Notebooks creados:     {len(all_notebooks)}
Fases completadas:     6 de 6
Archivos de datos:     {len(all_data)}
Tamaño total datos:    {total_size / (1024*1024):.1f} MB
Archivos de modelos:   {len(all_models)}

Técnicas aplicadas:
  - Análisis exploratorio de datos (EDA)
  - Limpieza y transformación de datos
  - Feature engineering
  - Pruebas estadísticas (t-test, ANOVA, chi²)
  - Regresión (Random Forest, Linear)
  - Clasificación (Logistic, RF, XGBoost)
  - Clustering (K-Means, DBSCAN)
  - Reducción de dimensionalidad (PCA)
  - Detección de anomalías (Z-score, Isolation Forest)
  - Series temporales (tendencias, estacionalidad)
  - Visualización estática (Matplotlib, Seaborn)
  - Visualización interactiva (Plotly)

Habilidades desarrolladas:
  - Python para ciencia de datos
  - Pensamiento estadístico
  - Machine Learning aplicado
  - Comunicación de resultados
  - Documentación técnica
  - Diseño de arquitecturas de datos
""")

RESUMEN FINAL DEL PROYECTO

Proyecto: Inteligencia de Flota Vehicular
──────────────────────────────────────────────────

Notebooks creados:     31
Fases completadas:     6 de 6
Archivos de datos:     34
Tamaño total datos:    966.5 MB
Archivos de modelos:   1

Técnicas aplicadas:
  - Análisis exploratorio de datos (EDA)
  - Limpieza y transformación de datos
  - Feature engineering
  - Pruebas estadísticas (t-test, ANOVA, chi²)
  - Regresión (Random Forest, Linear)
  - Clasificación (Logistic, RF, XGBoost)
  - Clustering (K-Means, DBSCAN)
  - Reducción de dimensionalidad (PCA)
  - Detección de anomalías (Z-score, Isolation Forest)
  - Series temporales (tendencias, estacionalidad)
  - Visualización estática (Matplotlib, Seaborn)
  - Visualización interactiva (Plotly)

Habilidades desarrolladas:
  - Python para ciencia de datos
  - Pensamiento estadístico
  - Machine Learning aplicado
  - Comunicación de resultados
  - Documentación técnica
  - Diseño de arquitecturas de datos



---

## Conclusión Final

### Lo que construimos

A lo largo de 6 fases y múltiples notebooks, hemos desarrollado un **sistema completo de inteligencia de flota vehicular** que demuestra el ciclo de vida completo de un proyecto de Ciencia de Datos:

1. **Entender el negocio**: Gestión de flotas vehiculares con telemetría IoT
2. **Explorar y preparar datos**: 50 vehículos, 30 días, 500 encuestas
3. **Analizar con rigor estadístico**: Pruebas de hipótesis, correlaciones, intervalos de confianza
4. **Modelar con Machine Learning**: Predicción de consumo, segmentación, clasificación de riesgo
5. **Detectar anomalías**: Mantenimiento predictivo, alertas tempranas
6. **Comunicar resultados**: Resumen ejecutivo, dashboard interactivo, documentación

### Competencias adquiridas

- **Técnicas**: Python, pandas, scikit-learn, Plotly, análisis estadístico
- **Metodológicas**: CRISP-DM, diseño experimental, validación de modelos
- **De negocio**: Traducción de datos a insights accionables, ROI de analítica
- **De ingeniería**: Arquitectura de producción, MLOps, documentación

### Siguientes pasos

1. **Corto plazo**: Implementar la API de predicción con FastAPI
2. **Medio plazo**: Configurar pipeline de streaming con Kafka
3. **Largo plazo**: Dashboard en Grafana/Metabase y CI/CD para modelos

---

### El viaje completo

```
Fase 1          Fase 2          Fase 3          Fase 4          Fase 5          Fase 6
Explorar   -->  Limpiar    -->  Analizar   -->  Modelar    -->  Detectar   -->  Comunicar
  "¿Qué         "¿Cómo          "¿Es            "¿Podemos       "¿Qué es        "¿Qué
  tenemos?"      preparamos?"    significativo?"  predecir?"     anormal?"       hacemos?"
```

Este proyecto demuestra que la Ciencia de Datos no es solo código y modelos: es un **proceso sistemático** para convertir datos en decisiones informadas.

---

*Proyecto de Ciencia de Datos - Fase 6: Documentación del Pipeline*

*Notebook final del proyecto.*